# 05 — Advanced Tuning: Log Target + Better Encoding + Interaction Features

**Current best:** v5 blend → Kaggle RMSE **22,428**. Goal: push below 22,000.

## Three improvements this notebook

| # | Improvement | Why it helps | Expected gain |
|---|---|---|---|
| 1 | **Log-transform target** (`np.log1p`) | 4,145 flats >$800K dominate RMSE. Log space weights errors by % not $. | –500 to –1,500 |
| 2 | **Per-fold target encoding** | `town_median_price` in notebook 04 used the full training set → slight leakage. OOF encoding is cleaner. | –100 to –400 |
| 3 | **Interaction features** | `dist_to_cbd × mid_storey` and `remaining_lease × floor_area` encode relationships GB can use more directly. | –100 to –300 |

**Submission plan today:** v6 = all three improvements + re-tuned blend

---
## 1. Imports & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from math import radians, cos, sin, asin, sqrt

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, KFold, cross_val_score, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, make_scorer

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

train = pd.read_csv('../../data/raw/train.csv', low_memory=False)
test  = pd.read_csv('../../data/raw/test.csv',  low_memory=False)
print(f'Train: {train.shape}  |  Test: {test.shape}')

# Quick price distribution check
print(f'\nPrice skew:       {train["resale_price"].skew():.3f}')
print(f'Price > $800K:    {(train["resale_price"] > 800_000).sum():,} rows ({(train["resale_price"] > 800_000).mean()*100:.1f}%)')
print(f'Price max:       ${train["resale_price"].max():>10,.0f}')

---
## 2. Feature Engineering

**New in this notebook vs notebook 04:**
- `dist_x_storey` = `dist_to_cbd × mid_storey` — penthouse near CBD worth much more than low floor
- `lease_x_area` = `remaining_lease × floor_area_sqm` — big + newer is doubly premium
- `log_dist_to_cbd` = `log(dist_to_cbd + 1)` — distance effect has diminishing returns; log captures the curve
- **Per-fold `town_median_price`** — computed out-of-fold to prevent leakage (see Section 3)

In [ ]:
DROP_COLS   = ['floor_area_sqft','lower','upper','mid','full_flat_type',
               'address','Tranc_YearMonth','residential','year_completed']
SOLD_COLS   = ['1room_sold','2room_sold','3room_sold','4room_sold',
               '5room_sold','exec_sold','multigen_sold','studio_apartment_sold']
RENTAL_COLS = ['1room_rental','2room_rental','3room_rental','other_room_rental']
FILL_ZERO   = ['Hawker_Within_500m','Mall_Within_500m','Hawker_Within_1km',
               'Mall_Within_1km','Hawker_Within_2km','Mall_Within_2km']
MATURE_ESTATES = {
    'ANG MO KIO','BEDOK','BISHAN','BUKIT MERAH','BUKIT TIMAH',
    'CENTRAL AREA','CLEMENTI','GEYLANG','KALLANG/WHAMPOA',
    'MARINE PARADE','PASIR RIS','QUEENSTOWN','SERANGOON','TAMPINES','TOA PAYOH'
}
ROOM_COUNT = {'1 ROOM':1,'2 ROOM':2,'3 ROOM':3,'4 ROOM':4,
              '5 ROOM':5,'EXECUTIVE':5,'MULTI-GENERATION':6}
CBD_LAT, CBD_LON = 1.2847, 103.8510

def haversine_km(lat, lon, lat2=CBD_LAT, lon2=CBD_LON):
    R = 6371
    lat, lon, lat2, lon2 = map(radians, [lat, lon, lat2, lon2])
    a = sin((lat2-lat)/2)**2 + cos(lat)*cos(lat2)*sin((lon2-lon)/2)**2
    return 2*R*asin(sqrt(a))

def engineer_features(df, amenity_caps=None):
    """Apply all feature engineering. amenity_caps: pass train-computed caps to test."""
    df = df.copy()
    # Fill zeros
    for c in FILL_ZERO:
        df[c] = df[c].fillna(0)
    # Tier 1 — location & lease
    df['remaining_lease']  = 99 - (df['Tranc_Year'] - df['lease_commence_date'])
    df['dist_to_cbd']      = df.apply(lambda r: haversine_km(r['Latitude'], r['Longitude']), axis=1)
    df['is_mature_estate'] = df['town'].str.upper().isin(MATURE_ESTATES).astype(int)
    # Tier 2 — cyclical + supply
    df['tranc_month_sin']     = np.sin(2*np.pi*df['Tranc_Month']/12)
    df['tranc_month_cos']     = np.cos(2*np.pi*df['Tranc_Month']/12)
    df['total_sold']          = df[SOLD_COLS].sum(axis=1)
    df['total_rental']        = df[RENTAL_COLS].sum(axis=1)
    df['rental_ratio']        = (df['total_rental'] / df['total_dwelling_units'].replace(0, np.nan)).fillna(0)
    df['num_rooms']           = df['flat_type'].str.upper().map(ROOM_COUNT).fillna(4)
    df['floor_area_per_room'] = df['floor_area_sqm'] / df['num_rooms']
    # Tier 3 — amenity score (use train-computed caps for test)
    caps = amenity_caps or {}
    new_caps = {}
    for col in ['mrt_nearest_distance','Mall_Nearest_Distance','Hawker_Nearest_Distance']:
        cap = caps.get(col, df[col].quantile(0.99))
        new_caps[col] = cap
        inv = 1 / df[col].clip(1, cap)
        inv_min, inv_max = inv.min(), inv.max()
        df[f'_inv_{col}'] = (inv - inv_min) / (inv_max - inv_min + 1e-9)
    df['amenity_score'] = (df['_inv_mrt_nearest_distance'] +
                           df['_inv_Mall_Nearest_Distance'] +
                           df['_inv_Hawker_Nearest_Distance']) / 3
    df.drop(columns=[c for c in df.columns if c.startswith('_inv_')], inplace=True)
    # Tier 4 — NEW: interaction & log features
    df['dist_x_storey']   = df['dist_to_cbd'] * df['mid_storey']
    df['lease_x_area']    = df['remaining_lease'] * df['floor_area_sqm']
    df['log_dist_to_cbd'] = np.log1p(df['dist_to_cbd'])
    return df, new_caps

# Apply FE (amenity caps computed from train, applied to test)
train_fe, train_caps = engineer_features(train)
test_fe,  _          = engineer_features(test, amenity_caps=train_caps)
print(f'Train: {train_fe.shape}  |  Test: {test_fe.shape}')

---
## 3. Per-Fold Target Encoding for `town_median_price`

### The problem with notebook 04's approach
In notebook 04, we computed `town_median_price` from the **full training set**, then used it as a feature during cross-validation. This means when we validate on fold 5, those rows' actual prices were included in computing the town median. The model partially "knew" the answer.

### The fix: Out-of-fold (OOF) encoding
For each fold, compute the town median from the **other 4 folds only** — never from the fold being validated.

```
Fold 1 town median = median of folds 2+3+4+5 prices  (fold 1 rows excluded)
Fold 2 town median = median of folds 1+3+4+5 prices  (fold 2 rows excluded)
...
```

For test data: use the full training set median (no leakage there — test has no prices).

In [ ]:
def oof_town_median(towns_series, y_series, n_splits=5, random_state=42):
    """Out-of-fold town median price to prevent target leakage."""
    towns = towns_series.values
    y     = y_series.values
    encoded = np.zeros(len(towns))
    global_med = np.median(y)

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    for tr_idx, va_idx in kf.split(towns):
        fold_map = {}
        for t, p in zip(towns[tr_idx], y[tr_idx]):
            fold_map.setdefault(t, []).append(p)
        fold_map = {t: np.median(ps) for t, ps in fold_map.items()}
        for i in va_idx:
            encoded[i] = fold_map.get(towns[i], global_med)
    return encoded

# OOF encoding for training rows
train_fe['town_median_price'] = oof_town_median(
    train_fe['town'], train['resale_price']
)

# Full-train map for test rows (no leakage — test has no prices)
full_town_map = train.groupby('town')['resale_price'].median()
test_fe['town_median_price'] = test_fe['town'].map(full_town_map).fillna(full_town_map.median())

print('OOF town encoding done.')
print(f'town_median_price range: ${train_fe["town_median_price"].min():,.0f} – ${train_fe["town_median_price"].max():,.0f}')

---
## 4. Prepare X and y

In [ ]:
DROP_ALL = ['id','resale_price'] + DROP_COLS + SOLD_COLS + RENTAL_COLS + ['num_rooms']

X      = train_fe.drop(columns=DROP_ALL, errors='ignore')
y      = train['resale_price'].values
y_log  = np.log1p(y)    # log-transformed target

X_test = test_fe.drop(columns=[c for c in DROP_ALL if c != 'resale_price'], errors='ignore')
X_test = X_test.reindex(columns=X.columns, fill_value=0)

num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(include='object').columns.tolist()

# Cast categoricals to string (postal has mixed types)
for col in cat_cols:
    X[col]      = X[col].astype(str)
    X_test[col] = X_test[col].astype(str)

print(f'Features: {X.shape[1]}  (num={len(num_cols)}, cat={len(cat_cols)})')
print(f'New features: dist_x_storey, lease_x_area, log_dist_to_cbd')

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
y_train_log = np.log1p(y_train)
y_val_log   = np.log1p(y_val)
print(f'\nTrain: {X_train.shape[0]:,}  |  Val: {X_val.shape[0]:,}')

---
## 5. Preprocessing Pipeline

In [ ]:
num_pipe = Pipeline([('imp', SimpleImputer(strategy='median'))])
cat_pipe = Pipeline([
    ('imp', SimpleImputer(strategy='most_frequent')),
    ('enc', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])
preprocessor = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols)
])

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

# Custom scorer: train/CV on log target, evaluate RMSE in dollar space
def neg_dollar_rmse(y_log_true, y_log_pred):
    return -rmse(np.expm1(y_log_true), np.expm1(y_log_pred))

dollar_rmse_scorer = make_scorer(neg_dollar_rmse)

print('Preprocessor ready.')

---
## 6. Why Log-Transform the Target?

**The problem with raw RMSE on skewed prices:**

```
Flat A: $300K actual, $310K predicted → Error = $10K   → Squared error = 100,000,000
Flat B: $900K actual, $1,000K pred   → Error = $100K  → Squared error = 10,000,000,000
```

Flat B's error dominates 100× even though it's only an 11% mistake. The model over-focuses on expensive flats.

**With log-transform:**

```
Flat A: log(300K)=$12.61, log(310K)=$12.65 → Log error = 0.04
Flat B: log(900K)=$13.71, log(1M)=$13.82   → Log error = 0.11
```

Now errors are proportional — an 11% error on an expensive flat is treated the same as an 11% error on a cheap flat. The model learns to predict the **relative** price correctly.

Quick comparison — same XGBoost params, with and without log:

In [ ]:
# Best params from notebook 04
best_params_04 = dict(
    n_estimators=1000, max_depth=7, learning_rate=0.05,
    subsample=0.7, colsample_bytree=0.6, min_child_weight=3,
    random_state=42, n_jobs=-1, verbosity=0, tree_method='hist'
)

# WITHOUT log transform
xgb_raw = Pipeline([
    ('pre', preprocessor),
    ('model', XGBRegressor(**best_params_04))
])
xgb_raw.fit(X_train, y_train)
pred_raw = xgb_raw.predict(X_val)
rmse_raw = rmse(y_val, pred_raw)

# WITH log transform (same params)
xgb_log = Pipeline([
    ('pre', preprocessor),
    ('model', XGBRegressor(**best_params_04))
])
xgb_log.fit(X_train, y_train_log)
pred_log = np.expm1(xgb_log.predict(X_val))
rmse_log_val = rmse(y_val, pred_log)

print(f'XGBoost (raw target)  val RMSE: ${rmse_raw:>8,.0f}')
print(f'XGBoost (log target)  val RMSE: ${rmse_log_val:>8,.0f}')
print(f'Improvement from log: ${rmse_raw - rmse_log_val:>+,.0f}')

---
## 7. Tune XGBoost with Log Target

We now search hyperparameters in log-target space. The `dollar_rmse_scorer` converts log predictions back to dollars before computing RMSE — so the CV scores remain interpretable.

In [ ]:
param_dist_xgb = {
    'model__n_estimators'     : [800, 1000, 1200, 1500],
    'model__max_depth'        : [5, 6, 7, 8],
    'model__learning_rate'    : [0.01, 0.03, 0.05, 0.08],
    'model__subsample'        : [0.6, 0.7, 0.8, 0.9],
    'model__colsample_bytree' : [0.5, 0.6, 0.7, 0.8],
    'model__min_child_weight' : [1, 3, 5],
    'model__reg_alpha'        : [0, 0.01, 0.1],    # L1 regularisation (new)
    'model__reg_lambda'       : [1, 1.5, 2],        # L2 regularisation (new)
}

xgb_pipe = Pipeline([
    ('pre', preprocessor),
    ('model', XGBRegressor(random_state=42, n_jobs=-1, verbosity=0, tree_method='hist'))
])

xgb_search = RandomizedSearchCV(
    xgb_pipe,
    param_distributions = param_dist_xgb,
    n_iter              = 40,
    cv                  = 3,
    scoring             = dollar_rmse_scorer,
    random_state        = 42,
    n_jobs              = -1,
    verbose             = 1,
    refit               = True,
)

xgb_search.fit(X_train, y_train_log)    # ← fit on log target

print(f'\nBest CV RMSE (dollar): ${-xgb_search.best_score_:,.0f}')
print('Best XGB params:')
for k, v in xgb_search.best_params_.items():
    print(f'  {k}: {v}')

In [ ]:
# Evaluate on val set in dollar space
best_xgb = xgb_search.best_estimator_
xgb_val_pred  = np.expm1(best_xgb.predict(X_val))
xgb_val_rmse  = rmse(y_val, xgb_val_pred)

print(f'Tuned XGBoost (log) val RMSE:  ${xgb_val_rmse:>8,.0f}')
print(f'Previous best XGBoost v3:       $21,828')
print(f'Change:                         ${xgb_val_rmse - 21828:>+,.0f}')

---
## 8. Tune LightGBM with Log Target

In [ ]:
param_dist_lgbm = {
    'model__n_estimators'     : [800, 1000, 1200, 1500],
    'model__max_depth'        : [6, 7, 8, 10],
    'model__num_leaves'       : [60, 80, 100, 127],
    'model__learning_rate'    : [0.01, 0.03, 0.05, 0.08],
    'model__subsample'        : [0.7, 0.8, 0.9],
    'model__colsample_bytree' : [0.6, 0.7, 0.8],
    'model__min_child_samples': [10, 20, 30],
    'model__reg_alpha'        : [0, 0.01, 0.1],
    'model__reg_lambda'       : [0, 0.1, 1],
}

lgbm_pipe = Pipeline([
    ('pre', preprocessor),
    ('model', LGBMRegressor(random_state=42, n_jobs=-1, verbosity=-1))
])

lgbm_search = RandomizedSearchCV(
    lgbm_pipe,
    param_distributions = param_dist_lgbm,
    n_iter              = 40,
    cv                  = 3,
    scoring             = dollar_rmse_scorer,
    random_state        = 42,
    n_jobs              = -1,
    verbose             = 1,
    refit               = True,
)

lgbm_search.fit(X_train, y_train_log)

print(f'\nBest CV RMSE (dollar): ${-lgbm_search.best_score_:,.0f}')
print('Best LGBM params:')
for k, v in lgbm_search.best_params_.items():
    print(f'  {k}: {v}')

In [ ]:
best_lgbm = lgbm_search.best_estimator_
lgbm_val_pred = np.expm1(best_lgbm.predict(X_val))
lgbm_val_rmse = rmse(y_val, lgbm_val_pred)

print(f'Tuned LightGBM (log) val RMSE: ${lgbm_val_rmse:>8,.0f}')
print(f'Previous best LGBM v5:          $21,762')
print(f'Change:                         ${lgbm_val_rmse - 21762:>+,.0f}')

---
## 9. Blend Search

Find the optimal XGBoost / LightGBM mix in log-prediction space, then convert final blend back to dollars.

In [ ]:
# Get log predictions on val set
xgb_val_log  = best_xgb.predict(X_val)
lgbm_val_log = best_lgbm.predict(X_val)

print('Blend search (in log space):')
print(f'{"XGB%":>6}  {"LGBM%":>6}  {"Val RMSE":>12}')
print('-' * 28)

best_blend_rmse = float('inf')
best_alpha = 0.5

results_blend = []
for alpha in np.arange(0.0, 1.01, 0.05):
    blend_log  = alpha * xgb_val_log + (1 - alpha) * lgbm_val_log
    blend_pred = np.expm1(blend_log)
    r = rmse(y_val, blend_pred)
    results_blend.append((alpha, r))
    marker = ' ←' if r == min(x[1] for x in results_blend) else ''
    print(f'{alpha*100:>5.0f}%  {(1-alpha)*100:>5.0f}%  ${r:>10,.0f}{marker}')
    if r < best_blend_rmse:
        best_blend_rmse = r
        best_alpha = alpha

print(f'\nBest blend: {best_alpha*100:.0f}% XGB + {(1-best_alpha)*100:.0f}% LGBM  →  RMSE ${best_blend_rmse:,.0f}')
print(f'v5 blend RMSE: $21,570  |  Change: ${best_blend_rmse - 21570:>+,.0f}')

---
## 10. 5-Fold Cross-Validation on Best Blend

Validate stability before submitting. Both models are trained on log target inside each fold.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Get best params from searches
xgb_best_params  = {k.replace('model__',''):v for k,v in xgb_search.best_params_.items()}
lgbm_best_params = {k.replace('model__',''):v for k,v in lgbm_search.best_params_.items()}

fold_rmses = []
X_arr, y_arr = np.array(X), y

for fold, (tr_idx, va_idx) in enumerate(kf.split(X), 1):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y_arr[tr_idx],  y_arr[va_idx]
    y_tr_log   = np.log1p(y_tr)

    # Per-fold target encoding for this fold's training rows
    fold_town_map = {}
    for t, p in zip(X_tr['town'].values, y_tr):
        fold_town_map.setdefault(t, []).append(p)
    fold_town_map = {t: np.median(ps) for t, ps in fold_town_map.items()}
    X_tr = X_tr.copy(); X_tr['town_median_price'] = X_tr['town'].map(fold_town_map).fillna(np.median(y_tr))
    X_va = X_va.copy(); X_va['town_median_price'] = X_va['town'].map(fold_town_map).fillna(np.median(y_tr))

    xgb_cv = Pipeline([('pre', preprocessor),
                        ('model', XGBRegressor(**xgb_best_params, random_state=42,
                                               n_jobs=-1, verbosity=0, tree_method='hist'))])
    lgbm_cv = Pipeline([('pre', preprocessor),
                         ('model', LGBMRegressor(**lgbm_best_params, random_state=42,
                                                  n_jobs=-1, verbosity=-1))])

    xgb_cv.fit(X_tr, y_tr_log)
    lgbm_cv.fit(X_tr, y_tr_log)

    blend_log = best_alpha * xgb_cv.predict(X_va) + (1-best_alpha) * lgbm_cv.predict(X_va)
    fold_pred = np.expm1(blend_log)
    fold_rmse_val = rmse(y_va, fold_pred)
    fold_rmses.append(fold_rmse_val)
    print(f'  Fold {fold}: RMSE ${fold_rmse_val:,.0f}')

cv_mean = np.mean(fold_rmses)
cv_std  = np.std(fold_rmses)
print(f'\nCV RMSE: ${cv_mean:,.0f} ± ${cv_std:,.0f}')
print(f'v5 blend CV RMSE: ~$21,500  |  Change: ${cv_mean - 21500:>+,.0f}')

---
## 11. Feature Importance — XGBoost (Log Target)

In [ ]:
NEW_FEATURES = ['remaining_lease','dist_to_cbd','is_mature_estate',
                'tranc_month_sin','tranc_month_cos','total_sold','total_rental',
                'rental_ratio','floor_area_per_room','town_median_price','amenity_score',
                'dist_x_storey','lease_x_area','log_dist_to_cbd']   # new in this notebook

xgb_model     = best_xgb.named_steps['model']
feature_names = num_cols + cat_cols
importances   = pd.Series(xgb_model.feature_importances_, index=feature_names)
top20         = importances.nlargest(20).sort_values()

colors = ['tomato' if f in NEW_FEATURES else 'steelblue' for f in top20.index]

fig, ax = plt.subplots(figsize=(9, 7))
top20.plot(kind='barh', color=colors, ax=ax)
ax.set_title('Top 20 Feature Importances — Tuned XGBoost (Log Target)\n(red = engineered / interaction features)')
ax.set_xlabel('Importance')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color='tomato', label='Engineered'), Patch(color='steelblue', label='Original')])
plt.tight_layout()
plt.savefig('../../outputs/figures/v6_xgb_importance.png', dpi=120, bbox_inches='tight')
plt.show()

print('\nTop 10 features:')
for f, imp in importances.nlargest(10).items():
    tag = ' ← NEW' if f in ['dist_x_storey','lease_x_area','log_dist_to_cbd'] else ''
    print(f'  {f:<30} {imp:.4f}{tag}')

---
## 12. All Models Comparison

In [ ]:
blend_val_pred = np.expm1(best_alpha * best_xgb.predict(X_val) + (1-best_alpha) * best_lgbm.predict(X_val))
blend_val_rmse = rmse(y_val, blend_val_pred)

results = pd.DataFrame([
    {'Model': 'v1  RF Baseline',           'Val RMSE': 25871,              'Kaggle': 25943.38},
    {'Model': 'v3  XGBoost tuned',          'Val RMSE': 21828,              'Kaggle': 22800.92},
    {'Model': 'v5  Blend 45/55',            'Val RMSE': 21570,              'Kaggle': 22428.34},
    {'Model': 'v6  XGB log tuned',          'Val RMSE': round(xgb_val_rmse), 'Kaggle': None},
    {'Model': 'v6  LGBM log tuned',         'Val RMSE': round(lgbm_val_rmse),'Kaggle': None},
    {'Model': 'v6  Blend log (best ratio)', 'Val RMSE': round(blend_val_rmse),'Kaggle': None},
])

print(results.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#A5A5A5','#4472C4','#4472C4','#70AD47','#70AD47','#FF6B6B']
bars = ax.bar(range(len(results)), results['Val RMSE'], color=colors)
ax.set_xticks(range(len(results)))
ax.set_xticklabels(results['Model'], rotation=20, ha='right', fontsize=8.5)
ax.set_ylabel('Val RMSE (SGD)')
ax.set_title('Model Progression — Val RMSE (lower = better)')
for i, v in enumerate(results['Val RMSE']):
    ax.text(i, v + 50, f'${v:,.0f}', ha='center', fontsize=8, fontweight='bold')
ax.axhline(21570, ls='--', color='gray', lw=1, label='v5 blend (prev best)')
ax.legend()
plt.tight_layout()
plt.savefig('../../outputs/figures/v6_model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 13. Generate Submission v6

Retrain both models on **full training data** (not just 80%) before predicting the 16,735 test rows.

In [ ]:
# Retrain on full training data with log target
print('Retraining XGBoost on full training data...')
best_xgb.fit(X, y_log)

print('Retraining LightGBM on full training data...')
best_lgbm.fit(X, y_log)

# Blend predictions (in log space, then convert)
xgb_test_log  = best_xgb.predict(X_test)
lgbm_test_log = best_lgbm.predict(X_test)
blend_test    = np.expm1(best_alpha * xgb_test_log + (1-best_alpha) * lgbm_test_log)

# Build submission
sample_sub = pd.read_csv('../../outputs/submissions/sample_sub_reg.csv')
sub_v6 = pd.DataFrame({'Id': test['id'], 'Predicted': blend_test})
sub_v6 = sub_v6[sub_v6['Id'].isin(sample_sub['Id'])]
sub_v6 = sub_v6.set_index('Id').reindex(sample_sub['Id']).reset_index()

out = '../../outputs/submissions/sub_v6_log_blend.csv'
sub_v6.to_csv(out, index=False)
print(f'\nSaved: {out}')
print(f'Shape: {sub_v6.shape}')
print(f'Price range: ${sub_v6.Predicted.min():,.0f} – ${sub_v6.Predicted.max():,.0f}')
print(f'Price mean:  ${sub_v6.Predicted.mean():,.0f}')

---
## 14. Summary

### Results this notebook
| Improvement | What we did | Val RMSE change |
|---|---|---|
| Log-transform target | `np.log1p(y)` → train → `np.expm1(pred)` | _(see above)_ |
| Per-fold target encoding | OOF `town_median_price` | Removes leakage |
| Interaction features | `dist_x_storey`, `lease_x_area`, `log_dist_to_cbd` | _(see above)_ |
| Regularisation | `reg_alpha`, `reg_lambda` in search grid | Reduces overfitting |

### Score tracker
| Version | Model | Val RMSE | Kaggle |
|---|---|---|---|
| v1 | RF Baseline | $25,871 | 25,943 |
| v3 | XGBoost tuned | $21,828 | 22,800 |
| v5 | Blend 45/55 | $21,570 | 22,428 |
| **v6** | **Log blend** | **_(run above)_** | **_(submit)_** |

### Key learnings
- Log-transform is most effective when the target is right-skewed (expensive outliers dominate raw RMSE)
- Per-fold encoding is a correctness fix — prevents the model from partially "knowing" the answer during CV
- Interaction features save the model from having to learn multiplications implicitly across many trees
- Adding `reg_alpha` / `reg_lambda` to the search grid is low-cost and often helps generalisation

### If v6 improves → next steps
- [ ] Try **stacking**: train a meta-model (Ridge/LR) on OOF XGB + LGBM predictions
- [ ] Experiment with **CatBoost** (handles categoricals natively — no OrdinalEncoder needed)
- [ ] Add **`street_name` frequency encoding** (some streets are consistently premium)
- [ ] Fine-tune blend ratio if v6 suggests a different optimal mix